# NB2 — Regression with Featured Data Only
**Key differences from NB1:**
- Featured engineered columns only (no raw stats)
- StandardScaler normalization
- Sample weighting: 2023/24=0.5, 2024/25=0.75, 2025/26=1.0
- 85/15 walk-forward split (GW15-29 of 2025/26 as test)
- No ensemble — best PyCaret model vs AutoKeras, winner selected by MAE

---
## ⚠️ Two Sessions Required
- **Session 1:** PyCaret
- **Session 2:** AutoKeras (restart runtime)

Cells 1-5 (Setup through Normalization) must be re-run in Session 2.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/fpl_thesis'
os.makedirs(f'{BASE}/data/processed/nb2_regression_featured', exist_ok=True)
os.makedirs(f'{BASE}/models/nb2_regression_featured', exist_ok=True)
print('Drive mounted and folders ready')

## 2. Load Featured Dataset

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = f'{BASE}/data/processed/featured_training_set.csv'
df = pd.read_csv(DATA_PATH)
df = df.sort_values(['element', 'season', 'GW']).reset_index(drop=True)
print(f'Loaded: {df.shape}')
print(df.head(3))

## 3. Define Features and Target

In [ ]:
# Metadata — not used in training
META_COLS = ['name', 'team', 'position', 'element', 'season', 'GW', 'total_points']

FEATURE_COLS = [c for c in df.columns if c not in META_COLS]
print(f'Feature columns ({len(FEATURE_COLS)}):')
print(FEATURE_COLS)

# Target: next GW points (shift -1 within each player-season)
df['target'] = df.groupby(['element', 'season'])['total_points'].shift(-1)
df = df.dropna(subset=['target']).reset_index(drop=True)
print(f'\nAfter target shift: {df.shape}')

# Filter out rows where player has no recent minutes (non-playing / rotation risk)
# This removes the constant 0-point rows that drag predictions down
# We keep all rows in the test set for honest evaluation
df_playing = df[df['rolling_mins_3'] > 0].copy()
print(f'After filtering non-players: {df_playing.shape}')
print(f'Removed {len(df) - len(df_playing)} rows ({(len(df)-len(df_playing))/len(df)*100:.1f}% of data)')

## 4. Walk-Forward Split (85/15)
- **Train:** 2023/24 + 2024/25 + GW1-14 of 2025/26
- **Test:** GW15-29 of 2025/26

In [ ]:
train = df_playing[~((df_playing['season'] == 2526) & (df_playing['GW'] >= 15))].copy()
test  = df_playing[  (df_playing['season'] == 2526) & (df_playing['GW'] >= 15)].copy()

print(f'Train: {len(train)} rows ({len(train)/len(df_playing)*100:.1f}%)')
print(f'Test:  {len(test)} rows ({len(test)/len(df_playing)*100:.1f}%)')

X_train = train[FEATURE_COLS]
y_train = train['target']
X_test  = test[FEATURE_COLS]
y_test  = test['target']

# Sample weights — recent seasons matter more
weight_map = {2324: 0.5, 2425: 0.75, 2526: 1.0}
sample_weights = train['season'].map(weight_map).values
print(f'\nSample weight counts:')
print(train['season'].value_counts().sort_index())

# Naive baseline
from sklearn.metrics import mean_absolute_error, r2_score
naive_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))
print(f'\nNaive baseline MAE: {naive_mae:.4f}')

In [ ]:
# Combined weights: season importance × playing time × target value
def get_combined_weight(row):
    season_w = {2324: 0.5, 2425: 0.75, 2526: 1.0}[row['season']]
    
    if row['rolling_mins_3'] < 30:
        return season_w * 0.3   # rarely plays — downweight
    elif row['target'] <= 1:
        return season_w * 0.5   # played but blanked
    elif row['target'] <= 5:
        return season_w * 1.0   # normal game
    elif row['target'] <= 9:
        return season_w * 2.0   # good game
    else:
        return season_w * 3.0   # haul

sample_weights = train.apply(get_combined_weight, axis=1).values

print('Combined weight distribution:')
import pandas as pd
w_series = pd.Series(sample_weights)
print(w_series.describe())
print(f'\nWeight sum: {sample_weights.sum():.0f}')

## 5. Imputation and Normalization

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import pickle

# Impute NaNs from rolling features at season start
imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

# Scale — fit on train only to prevent leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

# Save for dashboard use
with open(f'{BASE}/models/nb2_regression_featured/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open(f'{BASE}/models/nb2_regression_featured/imputer.pkl', 'wb') as f:
    pickle.dump(imputer, f)

print(f'X_train_scaled: {X_train_scaled.shape}')
print(f'X_test_scaled:  {X_test_scaled.shape}')
print('Scaler and imputer saved')

---
# SESSION 1 — PyCaret
Run fully, save outputs, then restart runtime for Session 2.

## 6. Install PyCaret

In [ ]:
!pip install pycaret==3.3.2 -q
print('PyCaret installed')

## 7. PyCaret Benchmarking

In [ ]:
import pandas as pd
import numpy as np
from pycaret.regression import *
from sklearn.metrics import mean_absolute_error, r2_score

# Build scaled DataFrame for PyCaret
train_pc = pd.DataFrame(X_train_scaled, columns=FEATURE_COLS)
train_pc['target'] = y_train.values

# Setup — disable internal preprocessing since we already scaled
s = setup(
    data               = train_pc,
    target             = 'target',
    session_id         = 42,
    normalize          = False,   # already done
    imputation_type    = None,    # already done
    data_split_shuffle = False,   # respect temporal order
    fold               = 5,
    verbose            = False,
)
print('PyCaret setup complete')

In [ ]:
# Compare all models sorted by MAE
best_models = compare_models(
    sort    = 'MAE',
    n_select= 3,
    exclude = ['ransac'],
    verbose = True,
)
print(f'\nTop 3 models: {[type(m).__name__ for m in best_models]}')

## 8. Tune and Evaluate Best PyCaret Model

In [ ]:
best = best_models[0]
print(f'Best model: {type(best).__name__}')

# Tune
tuned = tune_model(best, optimize='MAE', n_iter=20, verbose=False)

# Evaluate on test set
test_pc = pd.DataFrame(X_test_scaled, columns=FEATURE_COLS)
preds_pc = predict_model(tuned, data=test_pc)
pc_pred_values = np.clip(preds_pc['prediction_label'].values, 0, None)

mae_pc = mean_absolute_error(y_test, pc_pred_values)
r2_pc  = r2_score(y_test, pc_pred_values)
print(f'\nPyCaret Best Model Test Results:')
print(f'  MAE: {mae_pc:.4f}')
print(f'  R²:  {r2_pc:.4f}')
print(f'  vs Naive: {((naive_mae - mae_pc)/naive_mae)*100:.1f}% improvement')

## 9. Confidence Analysis — PyCaret

In [ ]:
# Analyse prediction distribution and confidence
import matplotlib.pyplot as plt

residuals = y_test.values - pc_pred_values
abs_errors = np.abs(residuals)

# Confidence bins based on predicted value ranges
bins = [0, 2, 4, 6, np.inf]
labels = ['0-2 pts', '2-4 pts', '4-6 pts', '6+ pts']
pred_bins = pd.cut(pc_pred_values, bins=bins, labels=labels)

conf_df = pd.DataFrame({
    'predicted': pc_pred_values,
    'actual': y_test.values,
    'abs_error': abs_errors,
    'pred_bin': pred_bins
})

print('PyCaret — MAE by prediction range:')
print(conf_df.groupby('pred_bin')['abs_error'].agg(['mean','std','count']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(pc_pred_values, y_test.values, alpha=0.3, s=5)
axes[0].plot([0, 20], [0, 20], 'r--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('PyCaret: Predicted vs Actual')
axes[1].hist(residuals, bins=50, edgecolor='black')
axes[1].set_xlabel('Residual'); axes[1].set_title('PyCaret: Residual Distribution')
plt.tight_layout()
plt.savefig(f'{BASE}/data/processed/nb2_regression_featured/pycaret_confidence.png', dpi=150)
plt.show()
print('Confidence plot saved')

## 10. Save PyCaret Model and Predictions

In [ ]:
import pickle

save_model(tuned, f'{BASE}/models/nb2_regression_featured/pycaret_best_model')

# Save predictions
preds_df = test[['name', 'team', 'position', 'element', 'season', 'GW']].copy().reset_index(drop=True)
preds_df['y_true']       = y_test.values
preds_df['pycaret_pred'] = pc_pred_values
preds_df.to_csv(f'{BASE}/data/processed/nb2_regression_featured/preds_session1.csv', index=False)

# Save metrics for comparison
metrics = {'naive_mae': naive_mae, 'mae_pc': mae_pc, 'r2_pc': r2_pc,
           'model_name': type(best).__name__}
import json
with open(f'{BASE}/data/processed/nb2_regression_featured/metrics_session1.json', 'w') as f:
    json.dump(metrics, f)

print('PyCaret model and predictions saved')
print('→ Restart runtime and run Session 2 (AutoKeras)')

---
# SESSION 2 — AutoKeras
⚠️ Restart runtime before this section.
Re-run cells 1–5 (Mount → Normalization) before continuing.

## 11. Install AutoKeras

In [ ]:
!pip install autokeras==3.0.0 tensorflow==2.18.0 -q
print('AutoKeras installed')

## 12. AutoKeras Neural Architecture Search

In [ ]:
import autokeras as ak
import tensorflow as tf
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

print(f'TensorFlow: {tf.__version__}')
print(f'AutoKeras:  {ak.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
ak_model = ak.StructuredDataRegressor(
    max_trials   = 10,
    overwrite    = True,
    directory    = f'{BASE}/models/nb2_regression_featured/ak_trials',
    project_name = 'nb2_autokeras',
    seed         = 42
)

ak_model.fit(
    X_train_scaled,
    y_train.values,
    epochs           = 35,
    validation_split = 0.1,
    verbose          = 1
)

## 13. Evaluate AutoKeras

In [ ]:
ak_preds = np.clip(ak_model.predict(X_test_scaled).flatten(), 0, None)

mae_ak = mean_absolute_error(y_test, ak_preds)
r2_ak  = r2_score(y_test, ak_preds)
print(f'AutoKeras Test MAE: {mae_ak:.4f}')
print(f'AutoKeras Test R²:  {r2_ak:.4f}')

## 14. Confidence Analysis — AutoKeras

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

residuals_ak = y_test.values - ak_preds
abs_errors_ak = np.abs(residuals_ak)

bins = [0, 2, 4, 6, np.inf]
labels = ['0-2 pts', '2-4 pts', '4-6 pts', '6+ pts']
pred_bins_ak = pd.cut(ak_preds, bins=bins, labels=labels)

conf_ak = pd.DataFrame({
    'predicted': ak_preds,
    'actual': y_test.values,
    'abs_error': abs_errors_ak,
    'pred_bin': pred_bins_ak
})

print('AutoKeras — MAE by prediction range:')
print(conf_ak.groupby('pred_bin')['abs_error'].agg(['mean','std','count']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(ak_preds, y_test.values, alpha=0.3, s=5)
axes[0].plot([0, 20], [0, 20], 'r--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('AutoKeras: Predicted vs Actual')
axes[1].hist(residuals_ak, bins=50, edgecolor='black')
axes[1].set_xlabel('Residual'); axes[1].set_title('AutoKeras: Residual Distribution')
plt.tight_layout()
plt.savefig(f'{BASE}/data/processed/nb2_regression_featured/autokeras_confidence.png', dpi=150)
plt.show()

## 15. Save AutoKeras Model

In [ ]:
best_ak = ak_model.export_model()
best_ak.save(f'{BASE}/models/nb2_regression_featured/autokeras_best_model.keras')
print('AutoKeras model saved')

# Add AutoKeras preds to session 1 predictions
preds_df = pd.read_csv(f'{BASE}/data/processed/nb2_regression_featured/preds_session1.csv')
preds_df['ak_pred'] = ak_preds
preds_df.to_csv(f'{BASE}/data/processed/nb2_regression_featured/all_preds.csv', index=False)
print('All predictions saved')

## 16. Model Comparison and Winner Selection

In [ ]:
import json

# Load session 1 metrics
with open(f'{BASE}/data/processed/nb2_regression_featured/metrics_session1.json') as f:
    m1 = json.load(f)

naive_mae  = m1['naive_mae']
mae_pc     = m1['mae_pc']
r2_pc      = m1['r2_pc']
model_name = m1['model_name']

print('='*50)
print('NB2 — FINAL MODEL COMPARISON')
print('='*50)
print(f'Naive Baseline MAE:      {naive_mae:.4f}')
print(f'PyCaret ({model_name}) MAE: {mae_pc:.4f}  R²: {r2_pc:.4f}')
print(f'AutoKeras MAE:           {mae_ak:.4f}  R²: {r2_ak:.4f}')
print()

if mae_pc <= mae_ak:
    winner = 'PyCaret'
    winner_mae = mae_pc
    winner_r2  = r2_pc
else:
    winner = 'AutoKeras'
    winner_mae = mae_ak
    winner_r2  = r2_ak

print(f'WINNER: {winner}')
print(f'  MAE: {winner_mae:.4f}')
print(f'  R²:  {winner_r2:.4f}')
print(f'  Improvement vs Naive: {((naive_mae-winner_mae)/naive_mae)*100:.1f}%')
print()
print('=== Comparison vs NB1 (all columns, no scaling) ===')
print(f'NB1 Huber MAE:     0.7969')
print(f'NB1 LightGBM MAE:  0.9301')
print(f'NB1 AutoKeras MAE: 0.9035')
print(f'NB2 Winner MAE:    {winner_mae:.4f}')

## 17. Generate Full Player Predictions (Winner Model)

In [ ]:
import pickle
import tensorflow as tf
import autokeras as ak

df_full = pd.read_csv(f'{BASE}/data/processed/featured_training_set.csv')
df_full = df_full.sort_values(['element', 'season', 'GW'])
latest  = df_full.groupby('element').last().reset_index()

META_COLS = ['name', 'team', 'position', 'element', 'season', 'GW', 'total_points']
FEATURE_COLS = [c for c in df_full.columns if c not in META_COLS]

X_latest = latest[FEATURE_COLS]

# Load scaler and imputer
with open(f'{BASE}/models/nb2_regression_featured/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open(f'{BASE}/models/nb2_regression_featured/imputer.pkl', 'rb') as f:
    imputer = pickle.load(f)

X_imp    = imputer.transform(X_latest)
X_scaled = scaler.transform(X_imp)
X_df     = pd.DataFrame(X_scaled, columns=FEATURE_COLS)

# Predict with winner
if winner == 'PyCaret':
    with open(f'{BASE}/models/nb2_regression_featured/pycaret_best_model.pkl', 'rb') as f:
        win_model = pickle.load(f)
    try:
        preds = np.clip(win_model.predict(X_df), 0, None)
    except:
        preds = np.clip(win_model.predict(X_scaled), 0, None)
else:
    win_model = tf.keras.models.load_model(
        f'{BASE}/models/nb2_regression_featured/autokeras_best_model.keras',
        custom_objects=ak.CUSTOM_OBJECTS
    )
    preds = np.clip(win_model.predict(X_scaled).flatten(), 0, None)

pos_map = {0:'DEF', 1:'FWD', 2:'GK', 3:'MID'}
out = latest[['name', 'team', 'position', 'element', 'value']].copy().reset_index(drop=True)
out['pos']           = latest['position_enc'].map(pos_map).values
out['predicted_pts'] = np.round(preds, 2)

out.to_csv(f'{BASE}/data/processed/nb2_regression_featured/predictions.csv', index=False)
print(f'Predictions saved: {len(out)} players')
print(f'Model used: {winner}')
print(out.sort_values("predicted_pts", ascending=False).head(10)[
    ['name','team','pos','predicted_pts','value']])